In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_error_map
from qiskit_aer import AerSimulator
from qiskit_aer import AerError
from qiskit_ibm_runtime import fake_provider
import seaborn as sns

In [ ]:
def oracle_1(qc):
    """
    Oracle for a constant function: f(x) = 0.
    This oracle does nothing, so the target qubit remains unchanged.
    """
    pass


def oracle_2(qc):
    """
    Oracle for a balanced function: f(x) = x.
    Flip the target qubit only when the input qubit is 1.
    """
    qc.cx(0, 1)


def oracle_3(qc):
    """
    Oracle for a balanced function: f(x) = x ⊕ 1.
    First apply f(x) = x, then flip the target qubit once more.
    """
    qc.cx(0, 1)
    qc.x(1)


def oracle_4(qc):
    """
    Oracle for a constant function: f(x) = 1.
    Always flip the target qubit regardless of the input.
    """
    qc.x(1)

## Select which oracle to use in the Deutsch algorithm

In [ ]:
oracle = oracle_1

## Createa the Deutsch circuit

In [ ]:
# Create a quantum circuit with 2 qubits and 1 classical bit
qc = QuantumCircuit(2, 1)

# Prepare the second qubit in the |1> state
qc.x(1)
qc.barrier()

# Apply a Hadamard gate to the second qubit to prepare the |->
# state, which is necessary for phase kickback
qc.h(1)
qc.barrier()

# Apply a Hadamard gate to the first qubit to create a superposition
# of |0> and |1>
qc.h(0)
qc.barrier()

# Apply the selected oracle
oracle(qc)
qc.barrier()

# Apply another Hadamard gate to the first qubit
# This causes interference that reveals whether the function
# is constant or balanced
qc.h(0)
qc.barrier()

# Measure the first qubit
qc.measure(0, 0)

## Use a fake IBM backend to simulate realistic device noise and coupling

In [ ]:
backend = fake_provider.FakeBelemV2()

### Visualize the error map of the backend

In [ ]:
plot_error_map(backend, figsize=(12, 8), show_title=False)

### Create a noisy simulator based on the fake backend

In [ ]:
simulator = AerSimulator.from_backend(backend)

In [ ]:
try:
    # Run the circuit 100 times and store individual measurement outcomes
    job = simulator.run(qc, shots=100, memory=True)
    
    # Retrieve the simulation result
    result = job.result()
except AerError as error:
    print(f'😱😱😱 {error}')

### Generate a preset transpiler pass manager optimized for the chosen backend
This pass manager applies a standard sequence of transpilation steps such as qubit layout selection, gate decomposition, routing, and circuit optimization.
The goal is to transform the circuit into a form that is executable on the selected backend while reducing errors and improving overall performance.

In [ ]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend
)

### Transpile the circuit so that it matches the backend architecture

In [ ]:
transpiled_qc = pass_manager.run(qc)

### Draw the transpiled circuit without showing idle wires

In [ ]:
transpiled_qc.draw("mpl", idle_wires=False)

In [ ]:
# Run the transpiled circuit 100 times and store individual measurement outcomes
job = simulator.run(transpiled_qc, shots=100, memory=True)

# Retrieve the simulation result
result = job.result()

# Get the raw measurement results as strings such as "0" or "1"
measurements = result.get_memory()
print(measurements)

# Interpret the measurement:
# "0" means the function is constant
# "1" means the function is balanced
measurements = [
    "constant" if each == "0" else "balanced"
    for each in measurements
]

# Plot the counts of constant vs balanced outcomes
sns.countplot(x=measurements)